In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver


from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict,Literal, Annotated, List
from pydantic import BaseModel,Field

### 1. LLM + embeddings

In [7]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [9]:
embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/Users/puneeth/Desktop/my_projects/Multi-Tool-Chatbot-LangGraph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. State

In [13]:
class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage],add_messages]

In [16]:
def chat_node(state: ChatState) -> ChatState:
    messages= state['messages']
    response= llm.invoke(messages)
    return {"messages": [response]} 

### 3. Workflow

In [ ]:
graph= StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)


### 4. CheckPointer

In [37]:
checkpointer= InMemorySaver()

In [38]:

workflow= graph.compile(checkpointer=checkpointer)


### 5. Invoking the Model

In [ ]:
config1= {"configurable":{"thread_id":"thread_1"}}


In [48]:
response= workflow.invoke({'messages':[HumanMessage(content="Lets Talk about persistance in 10 words")]}, config=config1)

In [49]:
response

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='1fdbc367-3046-4d94-8459-68f5ab1f3283'),
  AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CrUoGMd7hzctQVvEGsBKwo3LO28E3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b6169-5844-7f90-abca-454f43cd0194-0', usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
  HumanMessage(content

In [51]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='1fdbc367-3046-4d94-8459-68f5ab1f3283'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CrUoGMd7hzctQVvEGsBKwo3LO28E3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b6169-5844-7f90-abca-454f43cd0194-0', usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), Hu

In [52]:
config2= {"configurable":{"thread_id":"thread_2"}}

In [53]:
r= workflow.invoke({'messages':[HumanMessage(content="Explain LangGraph in 10 words")]}, config=config2)

In [54]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'messages': [HumanMessage(content='Explain LangGraph in 10 words', additional_kwargs={}, response_metadata={}, id='dca8a3c4-8a0a-4518-8570-6ab94885049d'), AIMessage(content='Language-based graph neural network for natural language processing tasks.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 15, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CrVInuyAyNMsXMH7XL0GPU9cdjGHH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b6186-3a03-7781-a60d-daa8d6e1a651-0', usage_metadata={'input_tokens': 15, 'output_tokens': 11, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cach